# 1. Sod shock tube

## The physical problem

The Sod shock tube is a one-dimensional experiment in which two uniform gas states are separated by a sharp discontinuity. Once the barrier is removed, pressure forces launch waves into both states. The problem is simple enough to have an analytic solution, but it exercises the essential ingredients of a compressible-fluid solver: reconstruction, Riemann fluxes, boundary conditions, and time integration.

This RadHydropy configuration uses a 4 cm Cartesian tube with an initially stationary, high-density/high-temperature gas outside a central low-density/low-temperature region. The central region occupies `1 < x < 3 cm`, so there are two mirrored Riemann problems at `x = 1 cm` and `x = 3 cm`. The analytic curve in the final figure is the solution for the left interface, mirrored about the centre of the tube. The periodic boundaries are sufficiently far away for this short run that the waves do not wrap around and interfere with the comparison.

The initial values are `rho_H = 1 g cm^-3`, `T_H = 1.550689...e-8 K`, `rho_L = 0.1 rho_H`, `T_L = 0.8 T_H`, and `u_H = u_L = 0`. With mean molecular weight `mu = 1`, the ideal-gas pressure is proportional to `rho T`, so the low state begins at `p_L = 0.08 p_H`. The adiabatic index is `gamma = 1.4`.

## Equations being solved

RadHydropy evolves the one-dimensional Euler equations in conservative form. Let `rho` be mass density, `u` velocity, `p` pressure, and `E` total energy density:

$$\frac{\partial}{\partial t}\begin{pmatrix}\rho\\\rho u\\E\end{pmatrix} + \frac{\partial}{\partial x}\begin{pmatrix}\rho u\\\rho u^2+p\\u(E+p)\end{pmatrix}=0.$$

The ideal-gas equation of state closes the system:

$$E = \frac{p}{\gamma-1}+\frac{1}{2}\rho u^2, \qquad p=(\gamma-1)\left(E-\frac{1}{2}\rho u^2\right).$$

In the finite-volume method, each cell stores an average of the conserved vector `U = (rho, rho u, E)`. During every timestep RadHydropy reconstructs states at cell faces, solves local Riemann problems to obtain numerical fluxes, and updates the cell averages using the flux difference. The CFL number controls the timestep so that information does not cross too many cells in one update.

## Analytic Sod solution

For one interface, the exact solution depends on the self-similar coordinate `xi = (x - x_I)/t`. The discontinuity separates a high state on the left (`rho_H, p_H, u_H`) from a low state on the right (`rho_L, p_L, u_L`). The solution contains four constant or smoothly varying regions separated by three waves:

1. a left-going rarefaction fan;
2. a contact discontinuity moving at the star velocity `u_*`; and
3. a right-going shock moving at speed `S`.

The unknown star pressure `p_*` is found from pressure and velocity matching across the two nonlinear waves. In this configuration the high-pressure side expands through a rarefaction, while the low-pressure side is compressed by a shock. Useful relations are

$$c_H=\sqrt{\gamma p_H/\rho_H}, \qquad \rho_{*,H}=\rho_H\left(\frac{p_*}{p_H}\right)^{1/\gamma},$$

$$u_* = \frac{2c_H}{\gamma-1}\left[1-\left(\frac{p_*}{p_H}\right)^{(\gamma-1)/(2\gamma)}\right]$$

for the rarefaction side, together with the shock relation on the low-pressure side. Once `p_*` and `u_*` are known, the rarefaction head, rarefaction tail, contact, and shock positions are obtained by multiplying their wave speeds by `t`. RadHydropy evaluates these relations in `shocktubecal()` and `shocktubeanalyticgraph()` in `example/SodShock1D/tools.py`.

The dashed curve in the generated figure is this analytic density profile. It is not an additional numerical run: it is a pointwise evaluation of the self-similar solution at the cell centres. Compare the numerical and analytic curves to identify shock smearing, contact smearing, and the effect of mesh resolution.

The notebook is designed for Google Colab. Run the setup cell once, then execute the cells from top to bottom.

In [ ]:
# Colab setup: download the code and install its plotting dependencies.
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPO = Path('/content/RadHydropy')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/tkc004/RadHydropy.git', str(REPO)], check=True)
PINNED = ['numpy==2.5.3', 'scipy==1.18.1', 'matplotlib==3.10.9', 'h5py==3.15.1', 'PyYAML==6.0.3', 'unyt==3.0.4']
PINNED_MARKER = Path('/content/.radhydropy_dependencies_pinned')
if not PINNED_MARKER.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINNED, '-e', '.[docs]'], cwd=REPO, check=True)
    PINNED_MARKER.touch()
    print('Dependencies installed. Restarting the Colab runtime once to load the pinned NumPy/SciPy binaries.')
    os.kill(os.getpid(), 9)
sys.path.insert(0, str(REPO))

%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (9, 5)

In [ ]:
# Run the maintained example in its own directory, as recommended by the docs.
from pathlib import Path
import subprocess
import sys

source_dir = REPO / 'example' / 'SodShock1D'
example_dir = Path('/content/radhydropy-sod')
if example_dir.exists():
    shutil.rmtree(example_dir)
example_dir.mkdir()
import yaml
config_data = yaml.safe_load((source_dir / 'sodshock1d.yaml').read_text())
config_data['par']['output']['directory'] = str(example_dir)
(example_dir / 'sodshock1d.yaml').write_text(yaml.safe_dump(config_data))
from example.SodShock1D import sodshock1d
sodshock1d.main(example_dir / 'sodshock1d.yaml')
print('Finished:', example_dir / 'SodShock1D.jpg')

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(example_dir / 'SodShock1D.jpg')))

## Questions to explore

- Where are the rarefaction, contact discontinuity, and shock?
- What changes if `--riemann-solver HLLC` is passed to the runner?
- How does the result change when the mesh resolution is reduced?

In [ ]:
# Optional: save results to Google Drive.
# from google.colab import drive
# drive.mount('/content/drive')
# save_dir = Path('/content/drive/MyDrive/RadHydropy/01_sod_shock')
# save_dir.mkdir(parents=True, exist_ok=True)
# for file in example_dir.glob('*'):
#     shutil.copy2(file, save_dir / file.name)
# print('Saved to:', save_dir)